# Machine Learning - Les 5: Evaluatiemetrieken, cross validation en grid search

Notebook bij les 5 van de leerlijn machine learning van S3 - AI.

© Auteur: Peter van den Berg

In de eerste drie machine learning lessen heb je verschillende supervised modellen leren trainen: een decision tree, een random forest en een k-nearest neighbors classifier (les 1 en 2), en lineaire en boom-gebaseerde regressiemodellen (les 3). In les 4 maakte je kennis met unsupervised learning via clustering.

Tot nu toe heb je classificatiemodellen vooral beoordeeld met de accuracy. In deze les gaan we dieper in op het evalueren van een classificatiemodel: je ontdekt waarom accuracy soms misleidt en je leert de metrieken **precision**, **recall** en de **F1-score** kennen.

Daarna pakken we twee technieken op die je in les 2 al in een simpele vorm hebt gezien, en maken we ze robuuster:
- **Cross validation** - een betrouwbaarder alternatief voor de enkele validatiesplit die je in les 2 gebruikte om `k` te tunen.
- **Grid search** - het automatiseren (en verbeteren) van de handmatige `for`-loop waarmee je in les 2 hyperparameters zocht.

### De dataset en het model

We werken in deze les met de **heart disease dataset**. Elke rij is een patiënt en de kolom `target` geeft aan of iemand een hartziekte heeft (`1`) of niet (`0`). Als model gebruiken we een **Random Forest**. Alles wat je hieronder leert werkt echter net zo goed voor de andere classifiers (decision tree, kNN).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [ ]:
url = 'https://raw.githubusercontent.com/mrdbourke/zero-to-mastery-ml/master/data/heart-disease.csv'
heart_df = pd.read_csv(url)
heart_df.head()

De kolom `target` is onze **target-variabele**. We kiezen bewust `1 = wel hartziekte` als de *positieve* klasse, precies zoals in de slides. Alle overige kolommen zijn de features.

Merk op dat de twee klassen ongeveer even groot zijn (redelijk *gebalanceerd*).

In [ ]:
X = heart_df.drop(columns='target')   # alle features
y = heart_df['target']                # 1 = wel hartziekte (positief), 0 = geen hartziekte

y.value_counts()

We splitsen in een train- en testset, met `stratify=y` zodat de klasseverdeling in beide sets gelijk blijft. De testset houden we apart tot de allerlaatste evaluatie.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print('Trainset:', X_train.shape[0], 'rijen')
print('Testset :', X_test.shape[0], 'rijen')

We trainen een Random Forest op de trainset en maken een voorspelling op de testset. Deze voorspellingen gebruiken we hierna om verschillende evaluatiemetrieken te berekenen.

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_pred[:15]

## Evaluatiemetrieken - classificatie

### Accuracy

Via `model.score(...)` gebruikte je al de **accuracy**: het aandeel correct voorspelde labels (het aantal goede voorspellingen gedeeld door het totaal). Naast `model.score(...)` heeft scikit-learn hiervoor ook de losse functie `accuracy_score`, zie https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

### Waarom accuracy misleidend kan zijn

Accuracy klinkt als een prima maat, maar hij kan flink **misleiden als de klassen niet in balans zijn**. Onze heart disease dataset is redelijk gebalanceerd, dus daar speelt dit minder, maar bekijk onderstaand voorbeeld:

> Stel je hebt een dataset met **990 niet-zieke** mensen en **10 zieke** mensen. Een lui model dat *altijd* 'niet ziek' voorspelt, heeft dan een accuracy van 990/1000 = **0.99**. Klinkt geweldig, maar het model herkent geen enkele zieke patiënt.

### Confusion matrix

Eén getal zoals accuracy verbergt *welke* fouten het model maakt. Een **confusion matrix** laat dat wel zien: het zet de voorspelde labels af tegen de werkelijke labels. Zo zie je precies hoeveel patiënten goed en fout geclassificeerd zijn, en in welke richting de fouten gaan. In de slides heb je precies zo'n confusion matrix voor hartziekte gezien. [Klik hier](https://hu-ai-s3-2026-ab.github.io/AI-S3-2026-AB-lesmateriaal-student/Visualisaties/classificatie-maten-visualisatie.html) hier voor een visualisatie over dit onderwerp.

We gebruiken `confusion_matrix` en `ConfusionMatrixDisplay`, zie https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['geen ziekte (0)', 'wel ziekte (1)'])
disp.plot(cmap='Blues')
plt.title('Confusion matrix')
plt.show()

### De bouwstenen: TN, FP, FN en TP

De vier vakjes van de confusion matrix hebben elk een naam. Ze vormen de bouwstenen voor bijna alle andere metrieken. Gezien vanuit de positieve klasse (`1 = wel ziekte`):

| | Voorspeld 0 | Voorspeld 1 |
|---|---|---|
| **Werkelijk 0** | True Negative (TN) | False Positive (FP) |
| **Werkelijk 1** | False Negative (FN) | True Positive (TP) |

- **True Negative (TN):** werkelijk gezond, ook zo voorspeld.
- **False Positive (FP):** werkelijk gezond, maar ten onrechte als ziek voorspeld (*vals alarm*).
- **False Negative (FN):** werkelijk ziek, maar gemist door het model.
- **True Positive (TP):** werkelijk ziek, ook zo voorspeld.

Met deze vier waarden kun je de metrieken uitdrukken:

$$\text{Accuracy} = \frac{TN + TP}{TN + FP + FN + TP} \qquad \text{Recall} = \frac{TP}{FN + TP} \qquad \text{Precision} = \frac{TP}{FP + TP}$$

Met `cm.ravel()` haal je de vier getallen uit de matrix (let op de volgorde: TN, FP, FN, TP):

In [ ]:
tn, fp, fn, tp = cm.ravel()
print('TN =', tn, '| FP =', fp, '| FN =', fn, '| TP =', tp)

### Recall en precision

$$\text{Recall / Sensitivity} = \frac{TP}{FN + TP} \qquad\qquad \text{Precision} = \frac{TP}{FP + TP}$$

**Recall** (ook wel *sensitivity*) beantwoordt de vraag: *welk deel van de werkelijk positieve gevallen heeft het model ook als positief herkend?* Een hoge recall betekent dat je weinig positieven mist (weinig false negatives). Dit wil je als het *missen* van een positief geval erg is - denk aan een gemiste ziekte.

**Precision** beantwoordt de vraag: *welk deel van de als positief voorspelde gevallen is daadwerkelijk positief?* Een hoge precision betekent weinig vals alarm (weinig false positives). Dit wil je als een *vals alarm* veel schade of overlast veroorzaakt.

Scikit-learn heeft `precision_score` en `recall_score`.

In [ ]:
from sklearn.metrics import precision_score, recall_score

print('Precision:', round(precision_score(y_test, y_pred), 3))
print('Recall   :', round(recall_score(y_test, y_pred), 3))

# Controle: hetzelfde als handmatig met de bouwstenen uit de confusion matrix
print('Handmatig precision:', round(tp / (tp + fp), 3))
print('Handmatig recall   :', round(tp / (fn + tp), 3))

### F1-score: de balans tussen precision en recall

Vaak is er een *trade-off* tussen precision en recall: maak je het model gretiger in het voorspellen van de positieve klasse, dan stijgt de recall maar daalt meestal de precision, en andersom. De **F1-score** vat beide samen in één getal, als het *harmonisch gemiddelde*:

$$F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

De F1-score ligt tussen 0 en 1: hoe dichter bij 1, hoe beter. Doordat het een harmonisch gemiddelde is, is de F1-score alleen hoog als *zowel* precision *als* recall hoog zijn. Het harmonisch gemiddelde laat de laagste score zwaarder meetellen dan het gewone gemiddelde. 

In [ ]:
from sklearn.metrics import f1_score

print(f1_score(y_test, y_pred))

In plaats van al deze metrieken los op te vragen, geeft `classification_report` in één keer een net overzicht voor beide klassen, zie https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred,
                            target_names=['geen ziekte (0)', 'wel ziekte (1)']))

### ML Opdracht 5.1 - Uitrekenen!

**Scenario:** je hebt een model getraind om spam te filteren. De testset bestaat uit:
- 980 e-mails die **geen** spam zijn (negatieve klasse, 0)
- 20 e-mails die **wel** spam zijn (positieve klasse, 1)

**Resultaten:**
- Het model classificeerde **15 van de 20 spam-mails correct** als spam.
- Het model classificeerde **6 van de 980 'geen spam'-mails onterecht** als spam.

a. Bepaal eerst met de hand de vier bouwstenen: TP, FP, FN en TN. Teken desnoods de confusion matrix erbij.

b. Bereken vervolgens met de hand de **accuracy**, **precision**, **recall** en de bijbehorende **F1-score**.

c. Controleer je antwoorden met code. Tip: bouw twee arrays `y_true` en `y_pred_spam` die precies deze situatie beschrijven, en gebruik daarna de metriek-functies die je hierboven hebt gezien.

### ML Opdracht 5.2 - Welke kies je?

Voor de onderstaande toepassingen: wil je vooral een **hoge recall** (geen positieven missen) of een **hoge precision** (weinig vals alarm)? Bedenk per geval wat de gevolgen zijn van een false negative en van een false positive, en beargumenteer je keuze.

- Frauduleuze transacties voorspellen
- Zwangerschapstest
- Voorspellen of vervolgonderzoek voor borstkanker nodig is
- Voorspellen of iemand vroegtijdig school zal verlaten
- Toeslagenaffaire: voorspellen of iemand die toeslagen aanvraagt fraudeert

*Bespreek je antwoorden met een medestudent. Zijn jullie het overal over eens? Sta bij het laatste voorbeeld extra stil bij de maatschappelijke gevolgen van fouten.*

### ML Opdracht 5.3 - Wat in je casus?

- Bedenk 2 use cases voor het model in jouw project-casus. Probeer een **ethische** en een **minder ethische** toepassing te verzinnen.
- Wil je in die toepassing een hoge recall of een hoge precision? Motiveer.

## Cross validation

### Het probleem met één enkele validatiesplit

In les 2 heb je hyperparameters getuned door de trainset op te splitsen in een train- en een validatieset. Daar liep je tegen een probleem aan: met een kleine dataset hangt de uitkomst sterk af van *welke* punten toevallig in de validatieset terechtkwamen. Een andere `random_state` gaf zomaar een andere 'beste' `k`. De schatting van de prestatie was dus onbetrouwbaar.

**k-fold cross validation** lost dit op. Het idee:
1. Verdeel de (train)data in *k* gelijke stukken (*folds*).
2. Train het model *k* keer. Elke keer gebruik je één fold als validatieset en de overige *k-1* folds om te trainen.
3. Je krijgt zo *k* scores. Het gemiddelde daarvan is een veel stabielere schatting, en de spreiding (standaarddeviatie) vertelt je hoe gevoelig het model is voor de keuze van de split.

$$\text{Performance} = \frac{1}{k}\sum_{i=1}^{k} \text{Performance}_i$$

We doen dit alleen op de trainset; de testset blijft apart voor de eerlijke eindevaluatie.

Laten we eerst met `StratifiedKFold` de folds zichtbaar maken. `StratifiedKFold` zorgt ervoor dat elke fold ongeveer dezelfde klasseverdeling heeft, wat bij classificatie belangrijk is. Zie https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f'Fold {i+1}: {len(train_idx)} train-voorbeelden, {len(val_idx)} validatie-voorbeelden')

De functie `cross_val_score` doet het hele proces in één regel: hij traint en evalueert het model *k* keer en geeft de *k* scores terug. Met `scoring` kies je zelf de metriek - hier de F1-score, waar je hierboven kennis mee hebt gemaakt. Zie https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html

> Voor een classifier gebruikt `cross_val_score` standaard al een gestratificeerde k-fold, dus je krijgt automatisch gebalanceerde folds.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='f1')

print('F1-score per fold:', np.round(scores, 2))
print('Gemiddelde F1    :', round(scores.mean(), 2))
print('Standaarddeviatie:', round(scores.std(), 2))

Je ziet dat de F1-score per fold verschilt - precies de variatie die je eerder met één enkele validatiesplit niet zag. Het gemiddelde geeft nu een betrouwbaardere schatting van de prestatie, en de standaarddeviatie laat zien hoe stabiel het model is. Dit is de robuuste manier om prestaties te vergelijken.

### ML Opdracht 5.4 - Cross validation uitproberen

a. Voer de cross validation hierboven nog eens uit met `scoring='accuracy'` en met `scoring='recall'`. Verschillen de conclusies die je zou trekken per metriek?

b. Probeer een paar waarden voor `cv` (bijvoorbeeld 3, 5 en 10). Wat gebeurt er met het gemiddelde en met de standaarddeviatie?

c. Vergelijk met cross validation een `RandomForestClassifier` en een `KNeighborsClassifier` (uit les 2) op deze data. Welk model presteert gemiddeld beter, en welk model is stabieler (kleinere standaarddeviatie)?

## Hyperparameter tuning met grid search

### Van handmatige for-loop naar grid search

In les 2 tunede je hyperparameters met een handgeschreven `for`-loop: je probeerde waarden voor `k` (of voor parameters van de random forest) één voor één uit. Dat werkt prima voor *één* hyperparameter, maar wordt al snel onhandig als je er meerdere tegelijk wilt combineren. Bovendien gebruikte je daarbij één vaste validatieset, met de onbetrouwbaarheid die je net hebt gezien.

Even ter herhaling: tijdens het `fit`-en leert een model zijn **parameters** zelf uit de data (bij een random forest bijvoorbeeld de splitsingscriteria in alle bomen). **Hyperparameters** kies je zelf *vóór* het trainen en bepalen *hoe* het model leert. Voor een `RandomForestClassifier` zijn dat er onder andere:
- `n_estimators`: het aantal bomen in het bos.
- `max_depth`: de maximale diepte per boom (tegen overfitting, ken je uit les 2).
- `min_samples_leaf`: het minimum aantal voorbeelden in een leaf-node.

### Grid search versus random search

Er zijn twee veelgebruikte zoekstrategieën, die je ook in de slides hebt gezien (met op de assen `n_estimators` en `max_depth`):

- **Grid search:** je geeft voor elke hyperparameter een lijstje waarden op en het algoritme probeert **alle combinaties** uit (een raster / *grid*). Grondig, maar het aantal combinaties - en dus de rekentijd - loopt snel op.
- **Random search:** je probeert een vast aantal **willekeurige** combinaties uit. Minder grondig, maar veel sneller en verrassend vaak bijna net zo goed, zeker als niet alle hyperparameters even belangrijk zijn.

We werken hier de grid search uit met `GridSearchCV`. Voor random search bestaat de vergelijkbare `RandomizedSearchCV`.

### GridSearchCV

`GridSearchCV` combineert de twee dingen uit deze les: het probeert alle hyperparameter-combinaties uit én evalueert elke combinatie met cross validation. Voor elke combinatie krijg je dus een gemiddelde score over de folds, en de combinatie met de hoogste score wint. Zo automatiseer je de for-loop uit les 2, én is de enkele onbetrouwbare validatiesplit verleden tijd.

Belangrijk: We laten `GridSearchCV` alleen los op de **trainset**. Zie https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

We definiëren eerst het *grid*: een dictionary met per hyperparameter de waarden die we willen proberen.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_leaf': [1, 3, 5],
}

# Aantal combinaties dat geprobeerd wordt (elk met 5-fold cross validation):
print(3 * 4 * 3, 'combinaties')

In [ ]:
grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,   # gebruik alle beschikbare processorkernen
)

grid.fit(X_train, y_train)

print('Beste hyperparameters:', grid.best_params_)
print('Beste gemiddelde F1 (cross validation):', round(grid.best_score_, 3))

Het getrainde `grid`-object heeft het beste model automatisch opnieuw op de hele trainset getraind. Je kunt er meteen mee voorspellen via `grid.best_estimator_` (of via `grid.predict`). Nu pas halen we de testset erbij, om te zien of het getunede model het echt beter doet dan het standaardmodel van het begin.

In [ ]:
best_model = grid.best_estimator_
y_pred_tuned = best_model.predict(X_test)

print('F1 standaardmodel (begin van dit notebook):', round(f1_score(y_test, y_pred), 3))
print('F1 getuned model                         :', round(f1_score(y_test, y_pred_tuned), 3))

Wil je zien hoe alle combinaties het deden? Dat zit in `grid.cv_results_`. Hieronder zetten we het in een dataframe en sorteren we op de gemiddelde score, van hoog naar laag.

In [ ]:
resultaten = pd.DataFrame(grid.cv_results_)
kolommen = ['param_n_estimators', 'param_max_depth', 'param_min_samples_leaf',
            'mean_test_score', 'std_test_score']
resultaten[kolommen].sort_values('mean_test_score', ascending=False).head(10)

> **Let op de rekentijd.** Het aantal combinaties is het product van alle lijstjes, en dat wordt nog eens vermenigvuldigd met het aantal folds. Ons grid van 3 x 4 x 3 = 36 combinaties met 5-fold CV betekent al 180 keer een random forest trainen. Voeg je nog een hyperparameter of meer waarden toe, dan groeit dit snel. Houd je grid dus behapbaar, of gebruik `RandomizedSearchCV` (https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) als het te groot wordt.

### ML Opdracht 5.5 - Tune je eigen model

a. Breid de `param_grid` uit met een extra hyperparameter van de `RandomForestClassifier` (bekijk de documentatie voor de mogelijkheden) of met andere waarden. Verbetert de F1-score op de testset?

b. Verander de `scoring` naar `'recall'`. Kiest de grid search nu andere hyperparameters? Kun je verklaren waarom?

c. Voer nu een grid search uit voor een `KNeighborsClassifier` in plaats van een random forest. Tune in ieder geval `n_neighbors` (de `k` uit les 2) en `weights`. Vergelijk het beste kNN-model met het beste random forest-model op de testset.

d. Pas dit toe op het model uit jouw eigen project-casus. Kies bewust een `scoring` die past bij het probleem (denk terug aan opdracht 5.2).

## De supervised learning workflow - compleet

Met wat je in deze lessenreeks hebt geleerd, ziet een zorgvuldige supervised learning-aanpak er zo uit:

1. Kies een of meer modeltypes (decision tree, random forest, kNN, lineaire regressie, ...).
2. Split de data in een trainset en een testset. Raak de testset daarna niet meer aan tot de allerlaatste evaluatie.
3. Prepareer de trainset.
4. Kies een evaluatiemetriek die past bij je probleem - accuracy, precision, recall of F1 bij classificatie; RMSE bij regressie.
5. Gebruik **cross validation** op de trainset voor een betrouwbare prestatieschatting, in plaats van één enkele validatiesplit.
6. Gebruik **grid search** (`GridSearchCV`) om binnen de trainset - met cross validation - de beste hyperparameters te vinden.
7. Prepareer de testset op dezelfde manier als de trainset.
8. Evalueer het uiteindelijke, getunede model één keer op de testset met je gekozen metriek.
    - Goed genoeg? Zet het model in op nieuwe data.
    - Niet goed genoeg? Verbeter (ander model, betere features, meer data, andere hyperparameters).


> **Tip voor regressie :** cross validation en grid search werken daar net zo goed. Kies dan alleen een passende `scoring`, bijvoorbeeld `scoring='neg_root_mean_squared_error'`, zodat je op de RMSE optimaliseert in plaats van op de F1-score.